In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Phan_Tich_Hanh_Vi_TMDT") \
    .master("spark://26.37.93.102:7077") \
    .config("spark.executor.memory", "3g") \
    .config("spark.sql.shuffle.partitions", "200") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

# Đọc dữ liệu sạch từ HDFS rồi đăng ký TempView
df = spark.read.parquet("hdfs://master:9000/data/test_cleaned.parquet")
df.createOrReplaceTempView("ecommerce_cleaned")
print("Đã load TempView ecommerce_cleaned\n")


In [ ]:
# CÂU 1 – PHÂN TÍCH PHỄU CHUYỂN ĐỔI
print("=" * 60)
print("CÂU 1 – PHỄU CHUYỂN ĐỔI")
print("=" * 60)

spark.sql("""
          WITH funnel AS (
              SELECT
                  ts_month,
                  cat_0,
                  -- Đếm session duy nhất có hành vi cart
                  COUNT(DISTINCT CASE WHEN event_type = 'cart'     THEN user_session END) AS cart_sessions,
                  -- Đếm session duy nhất có hành vi purchase
                  COUNT(DISTINCT CASE WHEN event_type = 'purchase' THEN user_session END) AS purchase_sessions,
                  -- Tổng session bất kể loại hành vi
                  COUNT(DISTINCT user_session) AS total_sessions
              FROM ecommerce_cleaned
              GROUP BY ts_month, cat_0
          )
          SELECT
              ts_month,
              cat_0,
              total_sessions,
              cart_sessions,
              purchase_sessions,
              -- Tỷ lệ thêm vào giỏ / tổng session
              ROUND(cart_sessions     / total_sessions     * 100, 2) AS cart_rate_pct,
              -- Tỷ lệ mua / thêm vào giỏ (điểm nghẽn chính)
              ROUND(purchase_sessions / NULLIF(cart_sessions, 0) * 100, 2) AS purchase_rate_pct
          FROM funnel
          ORDER BY ts_month, cat_0
          """).show(50, truncate=False)


In [ ]:
# CÂU 2 – TÌM CƠ HỘI BÁN CHÉO
print("=" * 60)
print("CÂU 2 – BÁN CHÉO (TOP 20 CẶP SẢN PHẨM)")
print("=" * 60)

spark.sql("""
          WITH purchases AS (
              -- Lọc riêng các sự kiện purchase
              SELECT user_session, product_id
              FROM ecommerce_cleaned
              WHERE event_type = 'purchase'
          )
          SELECT
              a.product_id AS product_a,
              b.product_id AS product_b,
              COUNT(*) AS co_purchase_count
          FROM purchases a
                   -- Self-join: cùng session, product_a < product_b để tránh lặp cặp đảo
                   JOIN purchases b
                        ON a.user_session = b.user_session
                            AND a.product_id < b.product_id
          GROUP BY a.product_id, b.product_id
          ORDER BY co_purchase_count DESC
              LIMIT 20
          """).show(20, truncate=False)


In [ ]:
# CÂU 3 – PHÂN KHÚC KHÁCH HÀNG THEO PARETO (80/20)
print("=" * 60)
print("CÂU 3 – PHÂN KHÚC PARETO (TOP 20% USER = 80% DOANH THU)")
print("=" * 60)

spark.sql("""
          WITH user_spend AS (
              -- Tổng chi tiêu của từng khách hàng
              SELECT
                  user_id,
                  SUM(price)             AS monetary,
                  COUNT(DISTINCT user_session) AS purchase_sessions
              FROM ecommerce_cleaned
              WHERE event_type = 'purchase'
              GROUP BY user_id
          ),
               ranked AS (
                   SELECT
                       user_id,
                       monetary,
                       purchase_sessions,
                       -- Xếp hạng từ người chi nhiều nhất xuống ít nhất
                       ROW_NUMBER() OVER (ORDER BY monetary DESC)  AS rnk,
                       COUNT(*)     OVER ()                        AS total_users,
                       SUM(monetary) OVER ()                       AS total_revenue,
            -- Doanh thu tích lũy từ top xuống
                       SUM(monetary) OVER (
                ORDER BY monetary DESC
                ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
            )                                           AS cumulative_revenue
                   FROM user_spend
               )
          SELECT
              user_id,
              ROUND(monetary, 2)                                    AS total_spend,
              purchase_sessions,
              rnk,
              total_users,
              ROUND(rnk * 100.0 / total_users, 2)                  AS user_pct,
              ROUND(cumulative_revenue * 100.0 / total_revenue, 2) AS cumulative_revenue_pct,
              -- Đánh dấu có thuộc nhóm top 20% Pareto không
              CASE
                  WHEN rnk * 1.0 / total_users <= 0.2 THEN 'Top 20% (Pareto)'
                  ELSE 'Remaining 80%'
                  END AS pareto_segment
          FROM ranked
          ORDER BY rnk
              LIMIT 50
          """).show(50, truncate=False)

# Tóm tắt Pareto: bao nhiêu user top 20% đóng góp bao nhiêu % doanh thu
spark.sql("""
          WITH user_spend AS (
              SELECT user_id, SUM(price) AS monetary
              FROM ecommerce_cleaned
              WHERE event_type = 'purchase'
              GROUP BY user_id
          ),
               ranked AS (
                   SELECT
                       user_id, monetary,
                       ROW_NUMBER() OVER (ORDER BY monetary DESC) AS rnk,
                       COUNT(*)      OVER ()                      AS total_users,
                       SUM(monetary) OVER ()                      AS total_revenue,
                       SUM(monetary) OVER (
                ORDER BY monetary DESC
                ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
            )                                          AS cumulative_revenue
                   FROM user_spend
               )
          SELECT
              CASE WHEN rnk * 1.0 / total_users <= 0.2 THEN 'Top 20%' ELSE 'Bottom 80%' END AS segment,
              COUNT(*)                                          AS user_count,
              ROUND(SUM(monetary), 2)                          AS segment_revenue,
              ROUND(SUM(monetary) * 100.0 / MAX(total_revenue), 2) AS revenue_pct
          FROM ranked
          GROUP BY CASE WHEN rnk * 1.0 / total_users <= 0.2 THEN 'Top 20%' ELSE 'Bottom 80%' END
          ORDER BY revenue_pct DESC
          """).show(truncate=False)


In [ ]:
# CÂU 4 – XU HƯỚNG DOANH THU VÀ TRUNG BÌNH ĐỘNG 3 NGÀY
print("=" * 60)
print("CÂU 4 – TRUNG BÌNH ĐỘNG 3 NGÀY & PHÁT HIỆN ĐỘT BIẾN")
print("=" * 60)

spark.sql("""
          WITH daily_revenue AS (
              -- Gom doanh thu theo ngày từ các đơn purchase
              SELECT
                  TO_DATE(event_time) AS event_date,
                  SUM(price)          AS daily_revenue,
                  COUNT(*)            AS order_count
              FROM ecommerce_cleaned
              WHERE event_type = 'purchase'
              GROUP BY TO_DATE(event_time)
          )
          SELECT
              event_date,
              ROUND(daily_revenue, 2) AS daily_revenue,
              order_count,
              -- Trung bình động 3 ngày (lùi 2 ngày + ngày hiện tại)
              -- Phù hợp với dataset 30 ngày: có giá trị ổn định từ ngày thứ 3
              ROUND(AVG(daily_revenue) OVER (
            ORDER BY event_date
            ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
        ), 2) AS ma_3day,
              -- Chênh lệch tuyệt đối so với MA3
              ROUND(daily_revenue - AVG(daily_revenue) OVER (
            ORDER BY event_date
            ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
        ), 2) AS delta_vs_ma,
              -- Flag đột biến: vượt 40% so với MA3 (ngưỡng thấp hơn vì cửa sổ ngắn hơn)
              CASE
                  WHEN daily_revenue > 1.4 * AVG(daily_revenue) OVER (
                ORDER BY event_date
                ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
            ) THEN 'SPIKE'
                  WHEN daily_revenue < 0.6 * AVG(daily_revenue) OVER (
                ORDER BY event_date
                ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
            ) THEN 'DROP'
                  ELSE 'Normal'
                  END AS anomaly_flag
          FROM daily_revenue
          ORDER BY event_date
          """).show(50, truncate=False)

In [ ]:
#  5 – TOP 3 THƯƠNG HIỆU CHỦ LỰC THEO NGÀNH HÀNG
print("=" * 60)
print("CÂU 5 – TOP 3 THƯƠNG HIỆU THEO NGÀNH HÀNG")
print("=" * 60)

spark.sql("""
          WITH brand_revenue AS (
              -- Tổng doanh thu theo ngành hàng và thương hiệu
              SELECT
                  cat_0,
                  brand,
                  SUM(price)   AS total_revenue,
                  COUNT(*)     AS order_count
              FROM ecommerce_cleaned
              WHERE event_type = 'purchase'
                AND brand != 'unknown'
          GROUP BY cat_0, brand
              )
          SELECT
              cat_0,
              brand,
              ROUND(total_revenue, 2) AS total_revenue,
              order_count,
              -- Xếp hạng doanh thu trong từng ngành, đồng hạng không bỏ số
              DENSE_RANK() OVER (
            PARTITION BY cat_0
            ORDER BY total_revenue DESC
        ) AS brand_rank
          FROM brand_revenue
                   -- Chỉ giữ top 3 mỗi ngành
                   QUALIFY DENSE_RANK() OVER (
        PARTITION BY cat_0
        ORDER BY total_revenue DESC
    ) <= 3
          ORDER BY cat_0, brand_rank
          """).show(50, truncate=False)


In [ ]:
# CÂU 6 – THỜI GIAN RA QUYẾT ĐỊNH MUA HÀNG
print("=" * 60)
print("CÂU 6 – THỜI GIAN TỪ GIỎ HÀNG → MUA")
print("=" * 60)

spark.sql("""
          WITH cart_events AS (
              -- Lấy thời điểm thêm giỏ sớm nhất của mỗi sản phẩm trong session
              SELECT user_id, product_id, user_session,
                     MIN(UNIX_TIMESTAMP(event_time)) AS cart_ts
              FROM ecommerce_cleaned
              WHERE event_type = 'cart'
              GROUP BY user_id, product_id, user_session
          ),
               purchase_events AS (
                   -- Lấy thời điểm purchase của cùng sản phẩm trong session
                   SELECT user_id, product_id, user_session,
                          MIN(UNIX_TIMESTAMP(event_time)) AS purchase_ts
                   FROM ecommerce_cleaned
                   WHERE event_type = 'purchase'
                   GROUP BY user_id, product_id, user_session
               )
          SELECT
              cat_0,
              -- Thời gian quyết định trung bình tính theo phút
              ROUND(AVG((p.purchase_ts - c.cart_ts) / 60.0), 2) AS avg_decision_minutes,
              ROUND(MIN((p.purchase_ts - c.cart_ts) / 60.0), 2) AS min_decision_minutes,
              ROUND(MAX((p.purchase_ts - c.cart_ts) / 60.0), 2) AS max_decision_minutes,
              COUNT(*)                                            AS sample_count
          FROM cart_events c
                   -- Chỉ ghép những session thực sự có purchase sau cart
                   JOIN purchase_events p
                        ON  c.user_id      = p.user_id
                            AND c.product_id   = p.product_id
                            AND c.user_session = p.user_session
                            AND p.purchase_ts  > c.cart_ts
                   JOIN ecommerce_cleaned e
                        ON  e.user_id    = c.user_id
                            AND e.product_id = c.product_id
          GROUP BY cat_0
          ORDER BY avg_decision_minutes
          """).show(truncate=False)


In [ ]:
# CÂU 7 – GIỎ HÀNG BỊ BỎ RƠI
print("=" * 60)
print("CÂU 7 – GIỎ HÀNG BỊ BỎ RƠI")
print("=" * 60)

spark.sql("""
          WITH cart_only AS (
              -- Session có cart nhưng không có bất kỳ purchase nào
              SELECT user_session
              FROM ecommerce_cleaned
              GROUP BY user_session
              HAVING SUM(CASE WHEN event_type = 'cart'     THEN 1 ELSE 0 END) > 0
                 AND SUM(CASE WHEN event_type = 'purchase' THEN 1 ELSE 0 END) = 0
          )
          SELECT
              e.cat_0,
              e.brand,
              COUNT(*)             AS abandoned_items,
              ROUND(SUM(e.price), 2) AS lost_revenue
          FROM ecommerce_cleaned e
                   -- Chỉ lấy các dòng thuộc session bỏ giỏ, event_type = cart
                   JOIN cart_only c ON e.user_session = c.user_session
          WHERE e.event_type = 'cart'
            AND e.brand != 'unknown'
          GROUP BY e.cat_0, e.brand
          ORDER BY lost_revenue DESC
              LIMIT 30
          """).show(30, truncate=False)


In [ ]:
# CÂU 8 – ĐỊNH VỊ GIÁ, PHÂN KHÚC THƯƠNG HIỆU & ĐỘ NHẠY GIÁ
print("=" * 60)
print("CÂU 8 – ĐỊNH VỊ GIÁ & ĐỘ NHẠY GIÁ")
print("=" * 60)

spark.sql("""
          WITH product_stats AS (
              -- Thống kê giá và chuyển đổi theo sản phẩm
              SELECT
                  product_id, cat_0, brand,
                  AVG(price)  AS avg_price,
                  SUM(CASE WHEN event_type = 'purchase' THEN 1 ELSE 0 END) AS purchases,
                  COUNT(*)    AS total_events,
                  ROUND(SUM(CASE WHEN event_type = 'purchase' THEN 1 ELSE 0 END)
                            / COUNT(*) * 100, 2) AS conversion_rate
              FROM ecommerce_cleaned
              GROUP BY product_id, cat_0, brand
          ),
               industry_avg AS (
                   -- Giá trung bình toàn ngành hàng (dùng CROSS JOIN để broadcast)
                   SELECT cat_0, AVG(avg_price) AS industry_avg_price
                   FROM product_stats
                   GROUP BY cat_0
               ),
               decile_tagged AS (
                   SELECT
                       p.*,
                       i.industry_avg_price,
                       -- Phân rã theo decile giá (1=rẻ nhất, 10=đắt nhất)
                       NTILE(10) OVER (ORDER BY p.avg_price ASC) AS price_decile,
            -- Phân loại định vị so với trung bình ngành
                       CASE
                           WHEN p.avg_price > 1.2 * i.industry_avg_price THEN 'Premium'
                           WHEN p.avg_price < 0.8 * i.industry_avg_price THEN 'Budget'
                           ELSE 'Mid-range'
                           END AS price_position
                   FROM product_stats p
                            JOIN industry_avg i USING (cat_0)
               )
          SELECT
              price_position,
              price_decile,
              COUNT(DISTINCT product_id)          AS product_count,
              ROUND(AVG(avg_price), 2)            AS avg_price,
              ROUND(AVG(conversion_rate), 2)      AS avg_conversion_rate_pct,
              SUM(purchases)                      AS total_purchases
          FROM decile_tagged
          GROUP BY price_position, price_decile
          ORDER BY price_position, price_decile
          """).show(50, truncate=False)


In [ ]:
# CÂU 9 – HÀNH TRÌNH KHÁCH HÀNG & HÀNH VI BẤT THƯỜNG
print("=" * 60)
print("CÂU 9A – NEXT BEST ACTION")
print("=" * 60)

spark.sql("""
          WITH session_events AS (
              SELECT
                  user_id, user_session, event_type,
                  UNIX_TIMESTAMP(event_time) AS ts,
                  -- Hành động kế tiếp trong cùng session, sắp xếp theo thời gian
                  LEAD(event_type) OVER (
                PARTITION BY user_id, user_session
                ORDER BY UNIX_TIMESTAMP(event_time)
            ) AS next_action
              FROM ecommerce_cleaned
          )
          SELECT
              event_type    AS current_action,
              next_action,
              COUNT(*)      AS transition_count,
              ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (PARTITION BY event_type), 2) AS transition_pct
          FROM session_events
          WHERE next_action IS NOT NULL
          GROUP BY event_type, next_action
          ORDER BY event_type, transition_count DESC
          """).show(truncate=False)

print("=" * 60)
print("CÂU 9B – CHUỖI NGÀY MUA LIÊN TIẾP (GAPS & ISLANDS)")
print("=" * 60)

spark.sql("""
          WITH purchase_days AS (
              -- Ngày mua duy nhất của từng user
              SELECT DISTINCT
                  user_id,
                  TO_DATE(event_time) AS purchase_date,
                  -- Số thứ tự ngày mua theo từng user
                  DENSE_RANK() OVER (
                PARTITION BY user_id
                ORDER BY TO_DATE(event_time)
            ) AS day_rank
              FROM ecommerce_cleaned
              WHERE event_type = 'purchase'
          ),
               islands AS (
                   -- Trừ rank cho date_diff để tạo group_id cho chuỗi liên tục
                   SELECT *,
                          DATE_SUB(purchase_date, day_rank) AS island_id
                   FROM purchase_days
               )
          SELECT
              user_id,
              MIN(purchase_date)  AS streak_start,
              MAX(purchase_date)  AS streak_end,
              COUNT(*)            AS streak_length_days
          FROM islands
          GROUP BY user_id, island_id
          HAVING COUNT(*) >= 3          -- Chỉ lấy chuỗi từ 3 ngày liên tiếp trở lên
          ORDER BY streak_length_days DESC
              LIMIT 20
          """).show(truncate=False)

print("=" * 60)
print("CÂU 9C – PHÁT HIỆN BOT")
print("=" * 60)

spark.sql("""
          SELECT
              user_session,
              user_id,
              SUM(CASE WHEN event_type = 'cart'     THEN 1 ELSE 0 END) AS cart_count,
              SUM(CASE WHEN event_type = 'purchase' THEN 1 ELSE 0 END) AS purchase_count,
              COUNT(DISTINCT product_id)                                 AS distinct_products
          FROM ecommerce_cleaned
          GROUP BY user_session, user_id
          -- Cart > 10 nhưng không có purchase nào → nghi ngờ bot cào giá
          HAVING cart_count > 10
             AND purchase_count = 0
          ORDER BY cart_count DESC
              LIMIT 30
          """).show(30, truncate=False)


In [ ]:
#  10 – BẢN ĐỒ NHIỆT DOANH THU THEO GIỜ & NGÀY
print("=" * 60)
print("CÂU 10 – BẢN ĐỒ NHIỆT & CHÊNH LỆCH DOANH THU THEO GIỜ")
print("=" * 60)

spark.sql("""
          WITH hourly_revenue AS (
              -- Doanh thu theo giờ và ngày trong tuần
              SELECT
                  ts_weekday,
                  ts_hour,
                  SUM(price)  AS revenue,
                  COUNT(*)    AS order_count
              FROM ecommerce_cleaned
              WHERE event_type = 'purchase'
              GROUP BY ts_weekday, ts_hour
          )
          SELECT
              ts_weekday,
              ts_hour,
              ROUND(revenue, 2)     AS revenue,
              order_count,
              -- Doanh thu giờ trước trong cùng ngày trong tuần
              ROUND(LAG(revenue) OVER (
            PARTITION BY ts_weekday
            ORDER BY ts_hour
        ), 2) AS prev_hour_revenue,
              -- Chênh lệch tuyệt đối so với giờ trước
              ROUND(revenue - LAG(revenue) OVER (
            PARTITION BY ts_weekday
            ORDER BY ts_hour
        ), 2) AS revenue_delta,
              -- Tăng trưởng phần trăm so với giờ trước
              ROUND((revenue - LAG(revenue) OVER (
            PARTITION BY ts_weekday
            ORDER BY ts_hour
        )) / NULLIF(LAG(revenue) OVER (
                      PARTITION BY ts_weekday
            ORDER BY ts_hour
                                 ), 0) * 100, 2) AS growth_pct
          FROM hourly_revenue
          ORDER BY ts_weekday, ts_hour
          """).show(100, truncate=False)


In [ ]:
print("\n Hoàn thành tất cả 10 câu phân tích.")
spark.stop()